# 01. Data Aggregation & Gold Layer Generation

Objetivo: Ingesta de datos (capa Silver), manejo de valores nulos y agregación temporal para consolidar una única fila por HCP (`NUEVO_ID`).

In [ ]:
import pandas as pd
import numpy as np
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def load_silver_data(file_path: str) -> pd.DataFrame:
    """
    Loads transactional silver layer data.
    """
    logger.info(f"Loading data from {file_path}")
    # return pd.read_csv(file_path)
    # Placeholder for actual data loading avoiding massive fake data generation
    return pd.DataFrame()


## Manejo de Nulos y Agregación

In [ ]:
def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Handles initial missing values defensively.
    """
    logger.info("Handling missing values.")
    df_clean = df.copy()
    
    # Fill numerical metrics with 0.0 robustly
    metric_cols = ['UC_TRX', 'UC_NBRX', 'BRAND1_NBRX', 'ORAL_NBRX', 'DETAILS', 'SAMPLES', 'RTE']
    existing_metric_cols = [c for c in metric_cols if c in df_clean.columns]
    
    if existing_metric_cols:
        df_clean[existing_metric_cols] = df_clean[existing_metric_cols].fillna(0.0)
    
    return df_clean

def aggregate_hcp_level(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregates time-series data (WEEK_ID, YEAR_QTR) to generate a single row per HCP (NUEVO_ID).
    """
    logger.info("Aggregating data to HCP level.")
    
    if df.empty:
        return df

    agg_dict = {
        'UC_TRX': 'sum',
        'UC_NBRX': 'sum',
        'BRAND1_NBRX': 'sum',
        'ORAL_NBRX': 'sum',
        'DETAILS': 'sum',
        'SAMPLES': 'sum',
        'RTE': 'sum',
        'ATSEG': 'first'  # Taking the first non-null label per HCP
    }
    
    valid_agg_dict = {k: v for k, v in agg_dict.items() if k in df.columns}
    
    if 'NUEVO_ID' not in df.columns:
        logger.warning("NUEVO_ID not found in dataset.")
        return df

    df_agg = df.groupby('NUEVO_ID').agg(valid_agg_dict).reset_index()
    return df_agg

if __name__ == '__main__':
    # df_silver = load_silver_data('data/silver_prescriptions.csv')
    # df_clean = handle_missing_values(df_silver)
    # df_gold = aggregate_hcp_level(df_clean)
    # df_gold.to_csv('data/gold_hcp_aggregated.csv', index=False)
    logger.info("Data aggregation skeleton completed.")
